# Stim → QIR

Compile [Stim](https://github.com/quantumlib/Stim) circuits to QIR and simulate them with `qdk.stim`.

- `stim.compile(src, None)` → `(qir, noise)`
- `stim.run(src, shots=..., type="clifford")` → per-shot results

In [ ]:
from qdk import stim
from qsharp_widgets import Histogram

## Basics

Compile a Bell pair to QIR.

In [ ]:
bell = """H 0
CX 0 1
MR 0 1
"""

qir, _ = stim.compile(bell, None)
print(qir)

In [ ]:
# Entangled: only 00 and 11 appear.
Histogram(stim.run(bell, shots=2000, type="clifford"), labels="kets")

## Preselection

`#!preselect_expect` retries until the measurement matches, so the recorded result is always that value — a single bar.

In [ ]:
# Force the measurement to 1: every shot reports 1.
preselect = """#!preselect_begin
H 0
MR 0
#!preselect_expect 0 1
"""
Histogram(stim.run(preselect, shots=2000, type="clifford"), labels="kets")

In [ ]:
## Noise channels

### Correlated error

`X0 X1` fire together → only `00` and `11`.

In [ ]:
correlated = """CORRELATED_ERROR(0.2) X0 X1
MR 0 1
"""
Histogram(stim.run(correlated, shots=2000, type="clifford"), labels="kets")

### Pauli error

Independent `X` on each qubit (p = 0.1).

In [ ]:
xerr = """X_ERROR(0.1) 0 1
MR 0 1
"""
Histogram(stim.run(xerr, shots=2000, type="clifford"), labels="kets")

### Loss

Lost qubits show up as `L` (p = 0.15).

In [ ]:
loss = """LOSS_ERROR(0.15) 0 1
MR 0 1
"""
Histogram(stim.run(loss, shots=2000, type="clifford"), labels="kets")

### Loss in a correlated error

Branches mix loss (`L`) and Pauli (`X`) terms.

In [ ]:
mixed = """CORRELATED_ERROR(0.1) L0
ELSE_CORRELATED_ERROR(0.1) L1
ELSE_CORRELATED_ERROR(0.1) L0 X1
MR 0 1
"""
Histogram(stim.run(mixed, shots=2000, type="clifford"), labels="kets")

### Depolarizing

`DEPOLARIZE1(p)`: one of 3 Paulis per qubit. `DEPOLARIZE2(p)`: one of 15 two-qubit Paulis.

In [ ]:
dep1 = """DEPOLARIZE1(0.3) 0 1
MR 0 1
"""
Histogram(stim.run(dep1, shots=2000, type="clifford"), labels="kets")

In [ ]:
dep2 = """DEPOLARIZE2(0.3) 0 1
MR 0 1
"""
Histogram(stim.run(dep2, shots=2000, type="clifford"), labels="kets")